# Setup, install RAGAS, load the test set

In [1]:
import sys, os
from pathlib import Path
import json
import logging

def find_project_root(marker="backend", start=None):
    current = Path(start or os.getcwd()).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / marker).is_dir():
            return candidate
    raise RuntimeError(f"Could not find a '{marker}' folder above {current}")

PROJECT_ROOT = find_project_root()
sys.path.insert(0, str(PROJECT_ROOT / "backend"))
sys.path.insert(0, str(PROJECT_ROOT / "backend" / "src"))

logging.basicConfig(level=logging.INFO, format="%(message)s")
logging.getLogger("httpx").setLevel(logging.WARNING)

from config.settings import settings
from medrag.embeddings.qdrant_client import get_qdrant_client
from medrag.retrieval.reranking import search_with_reranking
from medrag.generation.generation import generate_answer, DEFAULT_GENERATION_MODEL

import openai
from neo4j import GraphDatabase

qdrant = get_qdrant_client(settings.qdrant_url or "http://localhost:6333")
neo4j_driver = GraphDatabase.driver(settings.neo4j_uri, auth=(settings.neo4j_user, settings.neo4j_password))
openai_client = openai.OpenAI(api_key=settings.openai_api_key)

EVAL_DIR = PROJECT_ROOT / "data" / "eval"
with open(EVAL_DIR / "ragas_test_set.json", encoding="utf-8") as f:
    test_set = json.load(f)

print(f"Loaded {len(test_set)} test questions")

c:\Users\DELL\Desktop\medrag\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loaded 18 test questions


# Run the full pipeline on every test question

In [2]:
from medrag.retrieval.reranking import search_with_reranking

eval_records = []

for item in test_set:
    question = item["question"]
    print(f"Processing: {question}")

    results = search_with_reranking(qdrant, question, candidate_pool_size=20, top_n=5)
    contexts = [r["payload"]["raw_text"] for r in results]

    answer = generate_answer(
        question,
        qdrant_client=qdrant,
        openai_client=openai_client,
        neo4j_driver=neo4j_driver,
        model=DEFAULT_GENERATION_MODEL,
    )

    eval_records.append({
        "question": question,
        "answer": answer,
        "contexts": contexts,
        "ground_truth": item["ground_truth"],
    })

print(f"\nCollected {len(eval_records)} eval records")

Processing: What does metformin treat?


Loading sparse model 'Qdrant/bm25'...
Loading cross-encoder 'cross-encoder/ms-marco-MiniLM-L-6-v2'...
c:\Users\DELL\Desktop\medrag\venv\Lib\site-packages\huggingface_hub\file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Use pytorch device: cpu
Batches: 100%|██████████| 1/1 [00:12<00:00, 12.64s/it]


Processing: What are the contraindications of metformin?


Batches: 100%|██████████| 1/1 [00:04<00:00,  4.89s/it]


Processing: What is the recommended starting dose of metformin?


...

Processing: What are the long-term health outcomes studied for children conceived via IVF or ICSI?


Batches: 100%|██████████| 1/1 [00:11<00:00, 11.44s/it]


Processing: What barriers affect bone health awareness among lower-income women in China?


Batches: 100%|██████████| 1/1 [00:10<00:00, 10.58s/it]



Collected 18 eval records


# Build the RAGAS dataset, configure the judge LLM

In [3]:
import sys, types

_stub = types.ModuleType("langchain_community.chat_models.vertexai")

class ChatVertexAI:
    """Stub only - satisfies ragas's broken import (a confirmed, currently
    open upstream bug: ragas still imports ChatVertexAI from a path that
    was removed from langchain-community). We never use Google VertexAI,
    so this class is never actually instantiated."""
    def __init__(self, *args, **kwargs):
        raise NotImplementedError("VertexAI is not used in this project - stub only")

_stub.ChatVertexAI = ChatVertexAI
sys.modules["langchain_community.chat_models.vertexai"] = _stub

# Now the real imports should succeed
from datasets import Dataset
import ragas
print(f"ragas version: {ragas.__version__}")

from ragas import evaluate
from ragas.metrics import faithfulness, answer_relevancy, context_precision, context_recall
from ragas.llms import LangchainLLMWrapper
from langchain_openai import ChatOpenAI

eval_dataset = Dataset.from_list(eval_records)
print(eval_dataset)

judge_llm = LangchainLLMWrapper(ChatOpenAI(
    model=DEFAULT_GENERATION_MODEL,
    api_key=settings.openai_api_key,
))

print("RAGAS dataset and judge LLM ready")

ragas version: 0.3.9
Dataset({
    features: ['question', 'answer', 'contexts', 'ground_truth'],
    num_rows: 18
})
RAGAS dataset and judge LLM ready


C:\Users\DELL\AppData\Local\Temp\ipykernel_24612\2071409848.py:29: DeprecationWarning: LangchainLLMWrapper is deprecated and will be removed in a future version. Use llm_factory instead: from openai import OpenAI; from ragas.llms import llm_factory; llm = llm_factory('gpt-4o-mini', client=OpenAI(api_key='...'))
  judge_llm = LangchainLLMWrapper(ChatOpenAI(


# Run RAGAS evaluation

In [4]:
result = evaluate(
    eval_dataset,
    metrics=[faithfulness, answer_relevancy, context_precision, context_recall],
    llm=judge_llm,
)

print(result)

Evaluating:   0%|          | 0/72 [00:00<?, ?it/s]LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
Evaluating:   1%|▏         | 1/72 [00:02<03:22,  2.86s/it]LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
Evaluating:  72%|███████▏  | 52/72 [00:19<00:04,  4.71it/s]LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
Evaluating: 100%|██████████| 72/72 [00:30<00:00,  2.40it/s]

{'faithfulness': 0.8022, 'answer_relevancy': 0.8541, 'context_precision': 0.7833, 'context_recall': 1.0000}


# Inspect per-question scores, specifically the weak-coverage cases

In [5]:
df = result.to_pandas()

# Show every question with its per-metric scores
for idx, row in df.iterrows():
    print(f"[{idx}] {row['user_input'][:70]}")
    print(f"    faithfulness={row['faithfulness']:.4f}  answer_relevancy={row['answer_relevancy']:.4f}  "
          f"context_precision={row['context_precision']:.4f}  context_recall={row['context_recall']:.4f}")

[0] What does metformin treat?
    faithfulness=1.0000  answer_relevancy=0.9773  context_precision=1.0000  context_recall=1.0000
[1] What are the contraindications of metformin?
    faithfulness=1.0000  answer_relevancy=1.0000  context_precision=1.0000  context_recall=1.0000
[2] What is the recommended starting dose of metformin?
    faithfulness=1.0000  answer_relevancy=0.9728  context_precision=1.0000  context_recall=1.0000
[3] When should metformin be discontinued based on kidney function?
    faithfulness=1.0000  answer_relevancy=0.9554  context_precision=0.4778  context_recall=1.0000
[4] What effect can metformin have on vitamin B12 levels?
    faithfulness=1.0000  answer_relevancy=0.9868  context_precision=1.0000  context_recall=1.0000
[5] What is metformin-associated lactic acidosis characterized by?
    faithfulness=1.0000  answer_relevancy=0.9755  context_precision=1.0000  context_recall=1.0000
[6] What are the major side effects of ACE inhibitors?
    faithfulness=0.9231  ans

# Inspect the raw answers for the three zero-scoring rows

In [6]:
for idx in [11, 13, 14]:
    print(f"[{idx}] Q: {df.iloc[idx]['user_input']}")
    print(f"     A: {df.iloc[idx]['response']}")
    print()

[11] Q: Is there an association between diabetes and bone-health knowledge in women?
     A: The provided context does not contain specific information about the association between diabetes and bone-health knowledge in women.

[13] Q: What is the anatomical structure of the human heart's chambers?
     A: The provided context does not contain information about the anatomical structure of the human heart's chambers.

[14] Q: What is the general biological mechanism of the immune system's inflammatory response?
     A: The provided context does not contain specific information about the general biological mechanism of the immune system's inflammatory response.



# Check what retrieval actually pulled for question 11

In [7]:
print(df.iloc[11]['retrieved_contexts']) 

["Hypertension (β = -0.73; p < 0.001) and diabetes (β = -1.10; p < 0.001) were associated with slightly lower scores. Although most participants believed that bone health was important, women age 40 to 65 years with limited income in Shanghai had limited bone-health knowledge and generally did not take preventative steps that could have improved their bone health. These findings support practical, bone-health education in women's screening programs, community clinics, and chronic disease visits. These materials should be tailored to the anticipated reading levels of the populations served. Programs should explain calcium and vitamin D sources, weightbearing exercise, DEXA referral, and postfracture follow-up. Our findings may also be relevant to other urban and periurban populations in China and in other parts of Asia with similar social and economic characteristics.", 'Osteoporosis can cause painful, disabling, fatal fractures and impose substantial socioeconomic burdens. Because meno

# Test the same question with the fallback model

In [8]:
answer_nano = generate_answer(
    "Is there an association between diabetes and bone-health knowledge in women?",
    qdrant_client=qdrant, openai_client=openai_client, neo4j_driver=neo4j_driver,
    model="gpt-4.1-nano",
)
print("nano:", answer_nano)

answer_stronger = generate_answer(
    "Is there an association between diabetes and bone-health knowledge in women?",
    qdrant_client=qdrant, openai_client=openai_client, neo4j_driver=neo4j_driver,
    model="gpt-5.2",
)
print("\ngpt-5.2:", answer_stronger)

Batches: 100%|██████████| 1/1 [00:12<00:00, 12.05s/it]


nano: The provided context does not contain information about the association between diabetes and bone-health knowledge in women.


Batches: 100%|██████████| 1/1 [00:11<00:00, 11.55s/it]



gpt-5.2: Yes. In a survey of women aged 40–65 years with limited income in Shanghai, diabetes was associated with slightly lower bone-health knowledge scores (β = -1.10; p < 0.001). [1]


# Re-run evaluation with a stronger, still-affordable judge model

In [11]:
judge_llm_v2 = LangchainLLMWrapper(ChatOpenAI(
    model="gpt-5.2",
    api_key=settings.openai_api_key,
))

result_v2 = evaluate(
    eval_dataset,
    metrics=[faithfulness, answer_relevancy, context_precision, context_recall],
    llm=judge_llm_v2,
    raise_exceptions=True,
)

print(result_v2)

df_v2 = result_v2.to_pandas()
for idx, row in df_v2.iterrows():
    print(f"[{idx}] {row['user_input'][:70]}")
    print(f"    faithfulness={row['faithfulness']:.4f}  answer_relevancy={row['answer_relevancy']:.4f}  "
          f"context_precision={row['context_precision']:.4f}  context_recall={row['context_recall']:.4f}")

C:\Users\DELL\AppData\Local\Temp\ipykernel_24612\3738492950.py:1: DeprecationWarning: LangchainLLMWrapper is deprecated and will be removed in a future version. Use llm_factory instead: from openai import OpenAI; from ragas.llms import llm_factory; llm = llm_factory('gpt-4o-mini', client=OpenAI(api_key='...'))
  judge_llm_v2 = LangchainLLMWrapper(ChatOpenAI(
Evaluating:   3%|▎         | 2/72 [00:03<02:02,  1.75s/it]LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
Evaluating: 100%|██████████| 72/72 [00:53<00:00,  1.34it/s]

{'faithfulness': 0.6866, 'answer_relevancy': 0.7906, 'context_precision': 0.7738, 'context_recall': 0.7778}
[0] What does metformin treat?
    faithfulness=1.0000  answer_relevancy=0.9119  context_precision=1.0000  context_recall=1.0000
[1] What are the contraindications of metformin?
    faithfulness=1.0000  answer_relevancy=1.0000  context_precision=1.0000  context_recall=1.0000
[2] What is the recommended starting dose of metformin?
    faithfulness=0.8000  answer_relevancy=0.9728  context_precision=0.8333  context_recall=1.0000
[3] When should metformin be discontinued based on kidney function?
    faithfulness=1.0000  answer_relevancy=0.9237  context_precision=0.4167  context_recall=1.0000
[4] What effect can metformin have on vitamin B12 levels?
    faithfulness=1.0000  answer_relevancy=0.9162  context_precision=1.0000  context_recall=1.0000
[5] What is metformin-associated lactic acidosis characterized by?
    faithfulness=1.0000  answer_relevancy=0.9363  context_precision=1.000

# Inspect row 6's answer vs. ground truth directly

In [13]:
row = df_v2.iloc[6]
print("Ground truth:", row['reference'])
print("\nGenerated answer:", row['response'])

Ground truth: Major adverse effects of ACE inhibitors include dry cough, renal dysfunction in patients with impaired renal function, angioedema, hypotension, hyperkalemia, hypersensitivity reactions, and rarely cholestatic jaundice or hepatic failure.

Generated answer: The major adverse effects of ACE inhibitors include dry cough and renal dysfunction in patients with impaired renal function [1]. They can also cause angioedema, hypokalemia, and, rarely, neutropenia and agranulocytosis [2]. Additionally, ACE inhibitors may cause hypotension, dizziness, increased creatinine, hyperkalemia, and syncope, especially in volume- and salt-depleted patients [5].


# Check row 6's actual retrieved context

In [15]:
row = df_v2.iloc[6]
for i, ctx in enumerate(row['retrieved_contexts'], 1):
    print(f"[{i}] {ctx[:200]}")
    print()

[1] Class of drug | Major adverse effects
ACE inhibitors | dry cough, renal dysfunction in patients with impaired renal function
ARBs | increase in hepatic enzyme levels
CCBs (dihydropyridines) | headache

[2] 5. WARNINGS AND PRECAUTIONS ACE inhibitor use has been associated with the following: • Angioedema, with increased risk in patients with a prior history ( 5.1 ) • Hypotension and hyperkalemia ( 5.5 , 

[3] A few of the contraindications, such as use of ACE inhibitors and ARBs in pregnancy,
are absolute; most, however, are related to the fact that certain drugs could aggravate various
conditions. While c

[4] This is more likely to occur in patients with pre-existing renal impairment. Dosage reduction of ramipril and/or discontinuation of the diuretic may be required. 5.4 Neutropenia and Agranulocytosis In

[5] The following adverse reactions, mostly related to ACE inhibition, were reported more commonly in the high dose group: Table 1 Dose-related Adverse Drug Reactions: ATLAS tr

# Full, untruncated retrieved context for row 6

In [16]:
row = df_v2.iloc[6]
for i, ctx in enumerate(row['retrieved_contexts'], 1):
    print(f"[{i}] {ctx}")
    print("---")

[1] Class of drug | Major adverse effects
ACE inhibitors | dry cough, renal dysfunction in patients with impaired renal function
ARBs | increase in hepatic enzyme levels
CCBs (dihydropyridines) | headache, palpitation, rash, gravitational oedema
Diuretics (thiazide-like) | dry mouth, thirst, muscle cramps, impotence, hyperglycaemia,
hypercholesterolaemia, abnormality in electrolytes (hypokalaemia,
hypomagnesaemia, hypercalcaemia, hyponatraemia), pancreatitis
Beta-blockers | high-degree atrioventricular block, bradycardia, heart failure, Raynaud
phenomenon, impotence, fatigue, sleep disturbance including nightmares,
depression, alteration of lipid profi les
Alpha-blockers | orthostatic hypotension, syncope, dizziness, headache, drowsiness
Central alpha-agonist | orthostatic hypotension, bradycardia, drowsiness, dry mouth, galactorrhoea,
sexual dysfunction
Peripheral alpha-agonist
(reserpine) | depression, sedation, nasal stuffi ness
---
[2] 5. WARNINGS AND PRECAUTIONS ACE inhibitor use 

# Isolate context_recall on row 6, run multiple times to check consistency

In [18]:
row6 = eval_records[6]  # "What are the major side effects of ACE inhibitors?"

single_row_dataset = Dataset.from_list([row6])

for attempt in range(3):
    result_single = evaluate(
        single_row_dataset,
        metrics=[context_recall],
        llm=judge_llm_v2,
        raise_exceptions=True,
    )
    print(f"Attempt {attempt + 1}: context_recall = {result_single['context_recall']}")

Evaluating: 100%|██████████| 1/1 [00:03<00:00,  3.82s/it]


Attempt 1: context_recall = [0.0]


Evaluating: 100%|██████████| 1/1 [00:03<00:00,  3.23s/it]


Attempt 2: context_recall = [0.0]


Evaluating: 100%|██████████| 1/1 [00:03<00:00,  3.27s/it]

Attempt 3: context_recall = [1.0]
